In [1]:
# # 04 - Pelabelan Dataset dengan Label Sentimen 1.000 Data Tervalidasi
# Langkah:
# 1. Menentukan **label gold** dari 3 penilai (suara mayoritas). Kalimat yang tidak mencapai kesepakatan mayoritas **dikeluarkan dan tidak diadjudikasi** (arahan pembimbing: jumlah data total sudah besar), tetapi tetap dicatat dan dilaporkan di skripsi.
# 2. Melatih model **SVM** (konfigurasi sama dengan `03_klasifikasi_sentimen.py`) pada label gold.
# 3. Melabeli seluruh pasangan kalimat-aspek (~58 ribu pasangan dari ~71 ribu ulasan).
# 
# - Input : `hasil_validasi/label_penilai_1000.csv`, `data/aspek_kalimat.csv` (keluaran `02_ekstraksi_aspek.py`; satu baris = satu pasangan kalimat-aspek dari ulasan dataset)
# - Output: `hasil_validasi/label_gold_1000.csv`, `hasil_validasi/kalimat_dikeluarkan_tanpa_kesepakatan.csv`, `hasil_validasi/sentimen_kalimat_label_baru.csv`


# ===== SEL 1 (salin ke satu sel notebook) =====
import os, re, itertools, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

FOLDER_DATA  = "validasi data"           
FOLDER_HASIL = "hasil_validasi"   
os.makedirs(FOLDER_HASIL, exist_ok=True)

SEED = 42
LABEL = ["positif", "negatif", "netral"]
ASPEK = ["wifi", "harga", "suasana", "pelayanan"]
N_PENILAI = 3

def tafsir_kappa(k):
    """Skala Landis & Koch (1977)."""
    if k is None or np.isnan(k): return "tidak dapat dihitung"
    if k < 0:     return "buruk (di bawah kebetulan)"
    if k <= 0.20: return "sangat rendah"
    if k <= 0.40: return "rendah"
    if k <= 0.60: return "sedang"
    if k <= 0.80: return "baik (substantial)"
    return "sangat baik (almost perfect)"

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score


# ## 1. Label gold dari 3 penilai


# ===== SEL 2 (salin ke satu sel notebook) =====
sen = pd.read_csv(f"{FOLDER_HASIL}/label_penilai_1000.csv")
kolom = [f"penilai_{p}" for p in range(1, N_PENILAI + 1)]

def putuskan(baris):
    v = baris.dropna()
    if len(v) < 2: return "", "kurang_dari_2_penilai"
    hit = v.value_counts()
    if hit.iloc[0] == len(v):         return hit.index[0], "sepakat_semua"
    if hit.iloc[0] > len(v) / 2:      return hit.index[0], "mayoritas"
    return "", "tanpa_mayoritas"

h = sen[kolom].apply(putuskan, axis=1, result_type="expand"); h.columns = ["label_gold", "status"]
gold_all = pd.concat([sen, h], axis=1)
print(gold_all["status"].value_counts())
gold = gold_all[gold_all["label_gold"] != ""].copy()
perlu = gold_all[gold_all["label_gold"] == ""]
gold.to_csv(f"{FOLDER_HASIL}/label_gold_1000.csv", index=False)
perlu.to_csv(f"{FOLDER_HASIL}/kalimat_dikeluarkan_tanpa_kesepakatan.csv", index=False)
print(f"Label gold terpakai: {len(gold)} dari {len(gold_all)}; dikeluarkan (tanpa kesepakatan mayoritas): {len(perlu)} ({len(perlu)/len(gold_all):.1%})")
perlu[["id_aspek_kalimat", "aspek", "penilai_1", "penilai_2", "penilai_3", "kalimat_asli"]]
pd.crosstab(gold["aspek"], gold["label_gold"]).reindex(columns=LABEL)


# ## 2. Dataset pasangan kalimat-aspek


# ===== SEL 3 (salin ke satu sel notebook) =====
ak = pd.read_csv(f"{FOLDER_DATA}/aspek_kalimat.csv")
print("Jumlah pasangan kalimat-aspek:", len(ak), "| ulasan unik:", ak["id_ulasan"].nunique())
teks_model = lambda d: "aspek_" + d["aspek"] + " " + d["token_stem"].fillna("")
d = ak.merge(gold[["id_aspek_kalimat", "label_gold"]], on="id_aspek_kalimat")
assert len(d) == len(gold), "ada id gold yang tidak ada di aspek_kalimat.csv"
X, y = teks_model(d), d["label_gold"]


# ## 3. Melatih SVM pada label gold (split 80:20 + GridSearchCV, sama dengan 03_klasifikasi_sentimen.py)


# ===== SEL 4 (salin ke satu sel notebook) =====
strat = d["aspek"] + "_" + y
strat = strat.where(strat.map(strat.value_counts()) >= 10, "lain_" + y)   # strata terlalu kecil digabung per label
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=strat, random_state=SEED)
pipa = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
                 ("svm", SVC(class_weight="balanced", random_state=SEED))])
gs = GridSearchCV(pipa, {"svm__kernel": ["linear", "poly", "rbf"], "svm__C": [0.1, 1, 10]},
                  cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="f1_macro", n_jobs=-1)
gs.fit(Xtr, ytr)
print("Parameter terbaik:", gs.best_params_, "| F1-macro CV:", round(gs.best_score_, 4))
pred = gs.predict(Xte)
print(classification_report(yte, pred, labels=LABEL, digits=3, zero_division=0))


# ## 4. Melabeli seluruh pasangan kalimat-aspek
# Model dilatih ulang pada seluruh label gold, lalu memprediksi semua pasangan. Pasangan yang termasuk 1.000 data tetap memakai label gold (`sumber_label = manual_tervalidasi`).


# ===== SEL 5 (salin ke satu sel notebook) =====
final = gs.best_estimator_.fit(X, y)
hasil = ak[["id_aspek_kalimat", "id_kalimat", "id_ulasan", "id_cafe", "nama_cafe", "rating_bintang", "aspek", "kata_kunci", "kalimat_asli", "token_stem"]].copy()
hasil["sentimen"] = final.predict(teks_model(ak))
gmap = dict(zip(gold["id_aspek_kalimat"], gold["label_gold"]))
hasil["sumber_label"] = np.where(hasil["id_aspek_kalimat"].isin(gmap), "manual_tervalidasi", "model_svm")
m = hasil["sumber_label"] == "manual_tervalidasi"
hasil.loc[m, "sentimen"] = hasil.loc[m, "id_aspek_kalimat"].map(gmap)
hasil.to_csv(f"{FOLDER_HASIL}/sentimen_kalimat_label_baru.csv", index=False)
print(hasil["sumber_label"].value_counts())
pd.crosstab(hasil["aspek"], hasil["sentimen"]).reindex(columns=LABEL)


# ===== SEL 6 (salin ke satu sel notebook) =====
# Persentase sebaran sentimen per aspek
(pd.crosstab(hasil["aspek"], hasil["sentimen"], normalize="index").reindex(columns=LABEL) * 100).round(2)


# **Catatan**: label hasil model SVM adalah label prediksi (bukan penilaian manusia). Setelah label gold berubah, tabel `badge_aspek`, `fitur_idf`, `bobot_tfidf`, dan evaluasi relevansi perlu dibangun ulang dari `sentimen_kalimat_label_baru.csv`.

status
sepakat_semua      799
mayoritas          188
tanpa_mayoritas     13
Name: count, dtype: int64
Label gold terpakai: 987 dari 1000; dikeluarkan (tanpa kesepakatan mayoritas): 13 (1.3%)
Jumlah pasangan kalimat-aspek: 58643 | ulasan unik: 41478
Parameter terbaik: {'svm__C': 1, 'svm__kernel': 'linear'} | F1-macro CV: 0.6952
              precision    recall  f1-score   support

     positif      0.928     0.935     0.932       124
     negatif      0.875     0.933     0.903        60
      netral      0.556     0.357     0.435        14

    accuracy                          0.894       198
   macro avg      0.786     0.742     0.757       198
weighted avg      0.886     0.894     0.888       198

sumber_label
model_svm             57656
manual_tervalidasi      987
Name: count, dtype: int64


sentimen,positif,negatif,netral
aspek,,,
harga,79.92,15.66,4.42
pelayanan,77.06,22.60,0.33
suasana,94.82,5.07,0.11
wifi,55.08,24.18,20.75
